# SOH Forecasting — NASA Cell #23 (`Cell_23_NASA.csv`) — CC/CV decomposition

SOH is decomposed into a CC-phase contribution and a CV-phase contribution, each normalised by hardcoded RPT (Reference Performance Test) references:

$$\mathrm{SOH}_{cc}(\%) = W_{cc} \cdot \frac{Q_{cc} / \Delta V_{cc}}{2.47608}, \quad \mathrm{SOH}_{cv}(\%) = W_{cv} \cdot \frac{Q_{cv}}{0.830058}, \quad \mathrm{SOH} = \mathrm{SOH}_{cc} + \mathrm{SOH}_{cv}$$

Default weights `W_cc = W_cv = 50` give SOH ≈ 100 % for a fresh cell.

Differences from `soh_forecast_dqdv.ipynb`:
- Segments come from the data's `Original_Segment` column (no time-gap segmentation).
- No SOC. CC/CV split is done **by voltage threshold** (`V_CV = 4.199 V`).
- The SOH formula is applied to **every segment**, not just full ones.
- No smoothing, no interpolation. Segments with non-finite SOH (CC phase with `dv_cc == 0`) are dropped.

> Requires: `pandas numpy matplotlib scikit-learn torch`


In [ ]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

torch.manual_seed(0)
np.random.seed(0)

CSV_PATH = "Cell_23_NASA.csv"
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", DEVICE)


## 1. Load and inspect

In [ ]:
df = pd.read_csv(CSV_PATH)
df.columns = [c.strip() for c in df.columns]
# the first column is an unnamed index; drop or rename
if df.columns[0].startswith("Unnamed"):
    df = df.drop(columns=df.columns[0])
print("rows:", len(df), " columns:", list(df.columns))
print("TYPE values:", df["TYPE"].unique())
print("segments:", df["Original_Segment"].nunique(),
      "  range:", df["Original_Segment"].min(), "..", df["Original_Segment"].max())
df.head()


## 2. Per-segment aggregation with CC/CV split

We group by `Original_Segment` directly. Within each segment, samples below `V_CV = 4.199 V` are CC (voltage rising at constant current) and the remainder is CV (voltage held at ~4.2 V, current tapering). We compute `q_cc`, `dv_cc`, and `q_cv` per segment so the SOH formula can use them. A segment is dropped only if it has fewer than `MIN_ROWS` samples.


In [ ]:
MIN_ROWS = 10     # drop segments shorter than this many samples
V_CV     = 4.199  # V: threshold separating CC (below) from CV (above)

def aggregate_segment(g):
    if len(g) < MIN_ROWS:
        return None
    t_start = g["TIME"].iloc[0]
    t_end   = g["TIME"].iloc[-1]
    voltage = g["VOLTAGE"].values
    q_vals  = g["Q"].values

    # CC/CV split: first sample where V >= V_CV is the boundary
    above = np.where(voltage >= V_CV)[0]
    if len(above) > 0:
        trans = above[0]
        if trans > 0:
            q_cc  = float(q_vals[trans-1] - q_vals[0])
            dv_cc = float(voltage[trans-1] - voltage[0])
            q_cv  = float(q_vals[-1]      - q_vals[trans-1])
        else:
            q_cc, dv_cc = 0.0, 0.0
            q_cv = float(q_vals[-1] - q_vals[0])
    else:
        q_cc  = float(q_vals[-1]  - q_vals[0])
        dv_cc = float(voltage[-1] - voltage[0])
        q_cv  = 0.0

    return pd.Series({
        "t_start":        t_start,
        "t_end":          t_end,
        "duration_min":   (t_end - t_start) / 60.0,
        "v_start":        float(voltage[0]),
        "v_end":          float(voltage[-1]),
        "delta_v":        float(voltage[-1] - voltage[0]),
        "v_mean":         g["VOLTAGE"].mean(),
        "v_max":          g["VOLTAGE"].max(),
        "v_min":          g["VOLTAGE"].min(),
        "i_mean":         g["CURRENT"].mean(),
        "i_max":          g["CURRENT"].max(),
        "i_min":          g["CURRENT"].min(),
        "temp_mean":      g["TEMPERATURE"].mean(),
        "temp_max":       g["TEMPERATURE"].max(),
        "temp_min":       g["TEMPERATURE"].min(),
        "temp_spread":    g["TEMPERATURE"].max() - g["TEMPERATURE"].min(),
        "cap_charged_ah": float(q_vals[-1] - q_vals[0]),
        "energy_wh":      g["Energy_Wh"].iloc[-1] - g["Energy_Wh"].iloc[0],
        "n_samples":      len(g),
        "q_cc":           q_cc,
        "dv_cc":          dv_cc,
        "q_cv":           q_cv,
    })

records = []
for sid, g in df.groupby("Original_Segment", sort=True):
    rec = aggregate_segment(g)
    if rec is not None:
        rec["segment_id"] = sid
        records.append(rec)

cycles = pd.DataFrame(records).reset_index(drop=True)
cycles["cycle_idx"] = np.arange(len(cycles))
cycles["days_since_start"] = (cycles["t_start"] - cycles["t_start"].iloc[0]) / 86400.0
print(f"Kept {len(cycles)} segments out of {df['Original_Segment'].nunique()} (after MIN_ROWS filter)")
print("per-segment q_cc / dv_cc / q_cv distributions:")
print(cycles[['q_cc','dv_cc','q_cv']].describe())
cycles.head()


## 3. SOH from CC/CV decomposition

$$\mathrm{SOH}_{cc}(\%) = W_{cc} \cdot \frac{Q_{cc} / \Delta V_{cc}}{2.47608}, \quad \mathrm{SOH}_{cv}(\%) = W_{cv} \cdot \frac{Q_{cv}}{0.830058}$$

$$\mathrm{SOH}(\%) = \mathrm{SOH}_{cc} + \mathrm{SOH}_{cv}$$

Hardcoded RPT references and weights. The formula is applied **directly to every segment** — no smoothing, no interpolation, no full-segment filtering. Segments with non-finite SOH (CC phase with `dv_cc == 0`, or both `dv_cc == 0` and `q_cv == 0`) are dropped.


In [ ]:
REF_QCC_OVER_DVCC = 2.47608   # CC reference: Q_cc / dV_cc from RPT (Ah/V)
REF_QCV           = 0.830058  # CV reference: Q_cv from RPT (Ah)
W_CC = 50.0   # weight % for CC contribution
W_CV = 50.0   # weight % for CV contribution

# --- per-segment SOH components (formula applied to every segment) ---
cycles["soh_cc_pct"] = W_CC * (cycles["q_cc"] / cycles["dv_cc"]) / REF_QCC_OVER_DVCC
cycles["soh_cv_pct"] = W_CV * (cycles["q_cv"] / REF_QCV)
cycles["soh_pct"]    = cycles["soh_cc_pct"] + cycles["soh_cv_pct"]

# Drop segments with non-finite SOH (e.g. dv_cc == 0)
n_before = len(cycles)
cycles   = cycles[np.isfinite(cycles["soh_pct"])].reset_index(drop=True)
cycles["cycle_idx"] = np.arange(len(cycles))
print(f"Dropped segments with non-finite SOH: {n_before - len(cycles)}")
print(f"Segments remaining: {len(cycles)}")
print(cycles[['soh_cc_pct','soh_cv_pct','soh_pct']].describe())

plt.figure(figsize=(10, 4))
plt.plot(cycles["cycle_idx"], cycles["soh_cc_pct"], label=f"SOH_cc  (W_cc={W_CC:.0f})", alpha=0.7)
plt.plot(cycles["cycle_idx"], cycles["soh_cv_pct"], label=f"SOH_cv  (W_cv={W_CV:.0f})", alpha=0.7)
plt.plot(cycles["cycle_idx"], cycles["soh_pct"],    label="SOH = SOH_cc + SOH_cv", color="k", lw=1.5)
plt.xlabel("segment index"); plt.ylabel("SOH %"); plt.legend()
plt.title("SOH from CC/CV decomposition — NASA Cell #23 (every segment)")
plt.show()


## 4. Build windows and temporal split

In [ ]:
FEATURE_COLS = [
    "duration_min",
    "v_start", "v_end", "delta_v", "v_mean", "v_max", "v_min",
    "i_mean", "i_max", "i_min",
    "temp_mean", "temp_max", "temp_min", "temp_spread",
    "cap_charged_ah", "energy_wh", "n_samples",
    "q_cc", "dv_cc", "q_cv",
    "soh_cc_pct", "soh_cv_pct",
    "days_since_start",
    "soh_pct",   # current SOH is also an input feature for the next-segment forecast
]
WINDOW = 30
HORIZON = 1
TRAIN_FRAC = 0.75

X_all = cycles[FEATURE_COLS].values.astype(np.float32)
y_all = cycles["soh_pct"].values.astype(np.float32)

split = int(len(cycles) * TRAIN_FRAC)
print(f"Train segments 0..{split-1}  |  Test segments {split}..{len(cycles)-1}")

scaler = StandardScaler().fit(X_all[:split])
X_scaled = scaler.transform(X_all).astype(np.float32)

def make_windows(X, y, window, horizon, start, end):
    Xs, ys = [], []
    for i in range(start, end - window - horizon + 1):
        Xs.append(X[i:i+window])
        ys.append(y[i + window + horizon - 1])
    return np.array(Xs, dtype=np.float32), np.array(ys, dtype=np.float32)

X_train, y_train = make_windows(X_scaled, y_all, WINDOW, HORIZON, 0, split)
X_test,  y_test  = make_windows(X_scaled, y_all, WINDOW, HORIZON, split - WINDOW, len(cycles))
print("Train windows:", X_train.shape, "Test windows:", X_test.shape)


In [ ]:
class CycleWindowDS(Dataset):
    def __init__(self, X, y):
        self.X = torch.from_numpy(X); self.y = torch.from_numpy(y)
    def __len__(self): return len(self.X)
    def __getitem__(self, i): return self.X[i], self.y[i]

val_n = max(16, int(0.1 * len(X_train)))
train_ds = CycleWindowDS(X_train[:-val_n], y_train[:-val_n])
val_ds   = CycleWindowDS(X_train[-val_n:], y_train[-val_n:])
test_ds  = CycleWindowDS(X_test, y_test)

train_dl = DataLoader(train_ds, batch_size=32, shuffle=True)
val_dl   = DataLoader(val_ds,   batch_size=64)
test_dl  = DataLoader(test_ds,  batch_size=64)
print("train", len(train_ds), "val", len(val_ds), "test", len(test_ds))


## 5. Transformer encoder model

In [ ]:
class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=512):
        super().__init__()
        pe = torch.zeros(max_len, d_model)
        pos = torch.arange(max_len).unsqueeze(1).float()
        div = torch.exp(torch.arange(0, d_model, 2).float() * (-np.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(pos * div)
        pe[:, 1::2] = torch.cos(pos * div)
        self.register_buffer("pe", pe.unsqueeze(0))
    def forward(self, x): return x + self.pe[:, :x.size(1)]

class SOHTransformer(nn.Module):
    def __init__(self, n_features, d_model=64, nhead=4, layers=3, dim_ff=128, dropout=0.1):
        super().__init__()
        self.input_proj = nn.Linear(n_features, d_model)
        self.pos = PositionalEncoding(d_model)
        enc_layer = nn.TransformerEncoderLayer(
            d_model=d_model, nhead=nhead, dim_feedforward=dim_ff,
            dropout=dropout, batch_first=True, activation="gelu",
        )
        self.encoder = nn.TransformerEncoder(enc_layer, num_layers=layers)
        self.head = nn.Sequential(
            nn.Linear(d_model, d_model), nn.GELU(), nn.Dropout(dropout),
            nn.Linear(d_model, 1),
        )
    def forward(self, x):
        h = self.pos(self.input_proj(x))
        h = self.encoder(h)
        return self.head(h[:, -1]).squeeze(-1)

model = SOHTransformer(n_features=len(FEATURE_COLS)).to(DEVICE)
print(model)
print("Trainable params:", sum(p.numel() for p in model.parameters() if p.requires_grad))


## 6. Train

In [ ]:
EPOCHS = 80
LR = 1e-3
PATIENCE = 12

opt   = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-4)
sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=EPOCHS)
loss_fn = nn.MSELoss()

best_val = float("inf"); best_state = None; bad = 0
hist = {"train": [], "val": []}

for epoch in range(EPOCHS):
    model.train(); tr = 0.0; n = 0
    for xb, yb in train_dl:
        xb, yb = xb.to(DEVICE), yb.to(DEVICE)
        opt.zero_grad()
        loss = loss_fn(model(xb), yb)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        opt.step()
        tr += loss.item() * len(xb); n += len(xb)
    sched.step(); tr /= n

    model.eval(); vl = 0.0; nv = 0
    with torch.no_grad():
        for xb, yb in val_dl:
            xb, yb = xb.to(DEVICE), yb.to(DEVICE)
            vl += loss_fn(model(xb), yb).item() * len(xb); nv += len(xb)
    vl /= nv
    hist["train"].append(tr); hist["val"].append(vl)

    if vl < best_val - 1e-6:
        best_val = vl
        best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
        bad = 0
    else:
        bad += 1
    if epoch % 5 == 0 or epoch == EPOCHS - 1:
        print(f"epoch {epoch:3d}  train {tr:.5f}  val {vl:.5f}  best {best_val:.5f}")
    if bad >= PATIENCE:
        print(f"Early stop at epoch {epoch}, best val {best_val:.5f}")
        break

model.load_state_dict(best_state)

plt.figure(figsize=(7, 3))
plt.plot(hist["train"], label="train")
plt.plot(hist["val"],   label="val")
plt.yscale("log"); plt.xlabel("epoch"); plt.ylabel("MSE"); plt.legend(); plt.title("Loss")
plt.show()


## 7. Evaluate

Compare the Transformer against two baselines:
- **Persistence** — predict next SOH = current SOH (last position of the window).
- **Linear extrapolation** — least-squares linear fit on training segments.


In [ ]:
@torch.no_grad()
def predict(model, dl):
    model.eval()
    out = []
    for xb, _ in dl:
        out.append(model(xb.to(DEVICE)).cpu().numpy())
    return np.concatenate(out)

y_pred = predict(model, test_dl)

# Persistence: last SOH in each test window (undo standardisation)
soh_idx = FEATURE_COLS.index("soh_pct")
mu, sg = scaler.mean_[soh_idx], scaler.scale_[soh_idx]
y_persist = X_test[:, -1, soh_idx] * sg + mu

# Linear extrapolation fit on training segments
train_idx = np.arange(split)
coef = np.polyfit(train_idx, y_all[:split], 1)
test_cycle_idx = np.arange(split, len(cycles))[-len(y_test):]
y_linear = np.polyval(coef, test_cycle_idx)

def report(name, yt, yp):
    rmse = float(np.sqrt(mean_squared_error(yt, yp)))
    print(f"{name:14s}  MAE {mean_absolute_error(yt, yp):.4f}  RMSE {rmse:.4f}  R2 {r2_score(yt, yp):.4f}")

report("Transformer", y_test, y_pred)
report("Persistence", y_test, y_persist)
report("Linear",      y_test, y_linear)


In [ ]:
plt.figure(figsize=(10, 4))
plt.plot(cycles["cycle_idx"], cycles["soh_pct"], color="lightgray", label="full SOH series")
plt.plot(test_cycle_idx, y_test,    "k",     lw=2, label="test true")
plt.plot(test_cycle_idx, y_pred,    "C0",    label="Transformer")
plt.plot(test_cycle_idx, y_linear,  "C2--",  label="Linear")
plt.plot(test_cycle_idx, y_persist, "C3:",   label="Persistence")
plt.axvline(split, color="red", ls="--", lw=1, label="train/test split")
plt.xlabel("segment index"); plt.ylabel("SOH %")
plt.title("SOH forecast — NASA Cell #23 (dQ/ΔV, smoothed)")
plt.legend()
plt.show()


## Notes & next steps

- **Adjust weights** `W_CC`/`W_CV` in step 3 to change how CC and CV contribute. The defaults (50/50) put a fresh cell near 100 %.
- **`V_CV` threshold** controls where CC ends and CV begins. Tighten (e.g. 4.195) to capture more of the CC phase; loosen (4.2) to keep more in CV.
- **Negative `dv_cc`** can occur briefly from voltage drift; those segments produce non-finite SOH and are dropped.
- If you want SOH on a 0–100 scale that handles partial RPT segments differently, swap `W_CC`/`W_CV` to be capacity-weighted (e.g. `W_CC = 67.6`, `W_CV = 32.4`) using the RPT reference Ah split.
